In [17]:
import sys
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis")
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis\ucla-lapd")
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d, gaussian_filter, median_filter
from scipy.signal import find_peaks, savgol_filter

import read_hdf5 as rh
from bapsflib import lapd

from lp_analysis import find_sweep_indices, reshape_IV
from lp_iv_analysis import analyze_IV


%matplotlib qt
plt.rcParams.update({'font.size': 16})

In [2]:
ifn = r"D:\data\LAPD\Mar26-data\09-mach-xyplane-bias.hdf5"

# Extract directory and run number from ifn
data_dir = os.path.dirname(ifn)
run_num = os.path.basename(ifn).split('-')[0]  # "11" from "11-lp-sweep-xyplane-bias.hdf5"
# Create output filename
save_path = os.path.join(data_dir, f"{run_num}-sweep-data.npz")

with lapd.File(ifn) as f:
    rh.show_info(f)
    adc, digi_dict = rh.read_digitizer_config(f)
    pos_dict, xpos, ypos, zpos, npos, nshot = rh.read_bmotion_probe_motion(f)
    key = list(pos_dict.keys())[0]
    pos_array = pos_dict[key]

c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\maps\digitizers\siscrate.py:728: UserWarning: HDF5 structure unexpected...'siscf-6ch-mach-lp/SIS crate 3302 configurations[1]' does not define any valid channel numbers...not adding to `configs` dict
  warn(why)
c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\maps\digitizers\siscrate.py:728: UserWarning: HDF5 structure unexpected...'siscf-6ch-mach-lp/SIS crate 3302 configurations[2]' does not define any valid channel numbers...not adding to `configs` dict
  warn(why)


09-mach-xyplane-bias.hdf5 Overview
Generated by bapsflib (v0.0.0)
Generated date: 4/1/2026 11:25:35 AM


Filename:     09-mach-xyplane-bias.hdf5
Abs. Path:    D:\data\LAPD\Mar26-data\09-mach-xyplane-bias.hdf5
LaPD version: 1.2
Investigator: Jia Han
Run Date:     3/10/2026 6:54:01 PM

Exp. and Run Structure:
  (set)  Electrode_Biasing
  (exp)  +-- mar2026-jia
  (run)  |   +-- 09-mach-xyplane-bias

Run Description:
    Using 4 tips of a 6 tip uncoated shaft Mach probe for Vx,Vy measurements; record additional LP at P21 as reference signal.
    Most diagnostic and bias settings same as feb2026 experiments.
    
    
    LAPD B field:
    Black magnets at south: 888 A (PS12, 13),
    Yellow & Purple magnets: configured for uniform 800 G
    Except Yellow magnet PS 3, 4: 0 kG
    Black magnets at north: 0 A (PS11)
    (1600G-Source: 800 G: Bulk, 0 G: north end)
    
    South LaB6 source:
    He plasma, Discharge PS Voltage: 80 V, discharge current: 4.3 kA
    65 V cathode-anode voltage,   

In [3]:
with lapd.File(ifn) as f:
    data, tarr = rh.read_data(f, 1, 1, index_arr=slice(npos*nshot), adc=adc)
    Vx_arr = data['signal'].reshape((npos, nshot, -1))
    data, tarr = rh.read_data(f, 4, 6, index_arr=slice(npos*nshot), adc=adc)
    Vy_arr = data['signal'].reshape((npos, nshot, -1))
nt = len(tarr)

c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\maps\digitizers\siscrate.py:728: UserWarning: HDF5 structure unexpected...'siscf-6ch-mach-lp/SIS crate 3302 configurations[1]' does not define any valid channel numbers...not adding to `configs` dict
  warn(why)
c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\maps\digitizers\siscrate.py:728: UserWarning: HDF5 structure unexpected...'siscf-6ch-mach-lp/SIS crate 3302 configurations[2]' does not define any valid channel numbers...not adding to `configs` dict
  warn(why)
c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\utils\hdfreaddata.py:508: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  data = np.empty(shape, dtype=dtype)
c:\Users\hjia9\Docum

In [31]:
pndx = 500
plt.figure()
plt.title(f"Position {pos_array[pndx]}")
for i in range(1):
    sig = Vx_arr[pndx, i]
    filt = median_filter(sig, 5001)
    plt.plot(tarr, sig, alpha=0.5)
    plt.plot(tarr, filt)

plt.xlabel("Time")
plt.ylabel("Voltage")
plt.show()

In [ ]:
# Test on one signal
sig = Vx_arr[pndx, 0]
# ==========================================
# 1. Parameters & Dummy Data Setup
# ==========================================
fs = 1.0e6               
num_timepoints = 108544
median_window = 5001     # Large window 
dec_factor = 100          # Downsample factor


# ==========================================
# 2. Method A: Full Resolution (Brute Force)
# ==========================================
print("Running Full Resolution Median Filter...")
start_time_full = time.time()

# 1D median filter
baseline_full = median_filter(sig, size=median_window)

end_time_full = time.time()
time_full = end_time_full - start_time_full
print(f"Full Resolution took: {time_full:.4f} seconds")

# ==========================================
# 3. Method B: Decimate & Interpolate
# ==========================================
print("Running Decimate & Interpolate Filter...")
start_time_fast = time.time()

# 1. Decimate
x_orig = np.arange(num_timepoints)
x_dec = np.arange(0, num_timepoints, dec_factor)
signal_dec = sig[::dec_factor]

# 2. Scale window
small_window = median_window // dec_factor
if small_window % 2 == 0: small_window += 1

# 3. Median filter on small array
baseline_dec = median_filter(signal_dec, size=small_window)

# 4. Interpolate
baseline_fast = interp1d(x_dec, baseline_dec, fill_value="extrapolate")(x_orig)

end_time_fast = time.time()
time_fast = end_time_fast - start_time_fast
print(f"Decimate & Interpolate took: {time_fast:.4f} seconds")

# ==========================================
# 4. Calculate Difference
# ==========================================
speedup = time_full / time_fast if time_fast > 0 else float('inf')
print(f"--> Speedup Factor: {speedup:.1f}x faster")

difference = np.abs(baseline_full - baseline_fast)
max_error = np.max(difference)
print(f"--> Maximum amplitude difference: {max_error:.6f}")

# ==========================================
# 5. Visualization
# ==========================================
plt.figure(figsize=(14, 8))

# Top Plot: Overlay of both baselines
plt.subplot(2, 1, 1)
plt.plot(tarr, sig, color='gray', alpha=0.4, label='Raw Signal')
plt.plot(tarr, baseline_full, color='blue', linewidth=3, label='Full Resolution Baseline')
plt.plot(tarr, baseline_fast, color='red', linestyle='--', linewidth=2, label='Interpolated Baseline')
plt.title(f'Baseline Comparison (Speedup: {speedup:.1f}x)')
plt.ylabel('Amplitude')
plt.legend()
plt.grid(True)

# Bottom Plot: The Error (Difference)
plt.subplot(2, 1, 2)
plt.plot(tarr, difference, color='purple', label='Absolute Difference |Full - Interpolated|')
plt.title(f'Interpolation Error (Max Error: {max_error:.6f})')
plt.xlabel('Time (Seconds)')
plt.ylabel('Error Amplitude')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

Running Full Resolution Median Filter...
Full Resolution took: 0.0119 seconds
Running Decimate & Interpolate Filter...
Decimate & Interpolate took: 0.0022 seconds
--> Speedup Factor: 5.3x faster
--> Maximum amplitude difference: 0.029041
